# RF-DETR dataset2_augs — Predictions vs GT by Sequence
For each sequence in valid and test splits, show ALL frames side by side:
- **Left**: original image
- **Middle**: GT bounding boxes (green)
- **Right**: RF-DETR predictions (red)

In [ ]:
import os, re, warnings
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

warnings.filterwarnings('ignore')

DATASET_ROOT = Path('/home/dsa/stenosis/data/dataset2_split')
CHECKPOINT = '/home/dsa/stenosis/rfdetr_runs/dataset2_augs/checkpoint_best_ema.pth'
CONF_THRESHOLD = 0.15
CLASS_NAMES = {0: 'Stenosis'}

In [ ]:
# Load RF-DETR model
from rfdetr import RFDETRSmall

model = RFDETRSmall(num_classes=2, pretrain_weights=CHECKPOINT)
model.remove_optimized_model()
print('Model loaded.')

In [ ]:
def extract_sequence(fname: str) -> str:
    """14_012_3_0012_bmp_jpg.rf.xxx.jpg -> 14_012_3"""
    m = re.match(r'^(\d+_\d+_\d+)_', fname)
    return m.group(1) if m else None

def extract_frame_id(fname: str) -> str:
    m = re.search(r'_(\d{4})_bmp', fname)
    return m.group(1) if m else '?'

def load_yolo_boxes(label_path: Path, img_w: int, img_h: int):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().strip().splitlines():
        parts = line.split()
        cls = int(parts[0])
        cx, cy, w, h = map(float, parts[1:5])
        x1 = (cx - w / 2) * img_w
        y1 = (cy - h / 2) * img_h
        bw = w * img_w
        bh = h * img_h
        boxes.append((x1, y1, bw, bh, cls))
    return boxes

def gather_sequences(split: str):
    img_dir = DATASET_ROOT / split / 'images'
    seq_map = defaultdict(list)
    for f in sorted(img_dir.iterdir()):
        seq = extract_sequence(f.name)
        if seq:
            seq_map[seq].append(f)
    return dict(sorted(seq_map.items()))

def predict_batch(image_paths, batch_size=16):
    """Run predictions on a list of image paths in batches."""
    all_detections = []
    for i in range(0, len(image_paths), batch_size):
        batch = [str(p) for p in image_paths[i:i+batch_size]]
        dets = model.predict(batch, threshold=CONF_THRESHOLD)
        if not isinstance(dets, list):
            dets = [dets]
        all_detections.extend(dets)
    return all_detections

In [ ]:
def draw_boxes_gt(ax, boxes):
    for (x1, y1, bw, bh, cls) in boxes:
        rect = patches.Rectangle((x1, y1), bw, bh,
                                 linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)

def draw_boxes_pred(ax, det):
    mask = det.class_id == 0  # filter to stenosis class
    xyxy = det.xyxy[mask]
    confs = det.confidence[mask]
    for (x1, y1, x2, y2), conf in zip(xyxy, confs):
        bw, bh = x2 - x1, y2 - y1
        rect = patches.Rectangle((x1, y1), bw, bh,
                                 linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)

In [ ]:
def visualize_sequence(split: str, seq_name: str, files: list, detections: list):
    n = len(files)
    fig, axes = plt.subplots(n, 3, figsize=(18, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i, (img_path, det) in enumerate(zip(files, detections)):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        label_path = DATASET_ROOT / split / 'labels' / (img_path.stem + '.txt')
        gt_boxes = load_yolo_boxes(label_path, w, h)
        frame_str = extract_frame_id(img_path.name)

        # Col 0: original
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f'frame {frame_str}', fontsize=9)
        axes[i, 0].axis('off')

        # Col 1: GT
        axes[i, 1].imshow(img)
        draw_boxes_gt(axes[i, 1], gt_boxes)
        axes[i, 1].set_title(f'GT ({len(gt_boxes)} box)', fontsize=9)
        axes[i, 1].axis('off')

        # Col 2: predictions
        axes[i, 2].imshow(img)
        draw_boxes_pred(axes[i, 2], det)
        n_pred = int((det.class_id == 0).sum())
        axes[i, 2].set_title(f'Pred ({n_pred} box)', fontsize=9)
        axes[i, 2].axis('off')

    fig.suptitle(f'{split.upper()} — {seq_name}  ({n} frames)', fontsize=14, y=1.0)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## Validation split

In [ ]:
val_seqs = gather_sequences('valid')
print(f'Valid: {len(val_seqs)} sequences, {sum(len(v) for v in val_seqs.values())} images total')

for seq_name, files in val_seqs.items():
    print(f'  Running inference on {seq_name} ({len(files)} frames)...')
    dets = predict_batch(files)
    visualize_sequence('valid', seq_name, files, dets)

## Test split

In [ ]:
test_seqs = gather_sequences('test')
print(f'Test: {len(test_seqs)} sequences, {sum(len(v) for v in test_seqs.values())} images total')

for seq_name, files in test_seqs.items():
    print(f'  Running inference on {seq_name} ({len(files)} frames)...')
    dets = predict_batch(files)
    visualize_sequence('test', seq_name, files, dets)